<a href="https://colab.research.google.com/github/moneet-dev/nanowire-ml/blob/main/notebooks/nanowire_ml_full_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NanowireML — Full Pipeline

Reproducing and extending Raya et al. 2025: sequence-only ML prediction of microbial nanowire proteins.

**Run all cells** — the notebook handles setup, feature extraction, baseline reproduction,
hyperparameter tuning, calibration, SHAP, feature ablation, and the autoresearch loop.

Features are persisted to Google Drive so reconnecting doesn't require re-extraction.

> Tip: Use a **T4 GPU runtime** (Runtime → Change runtime type → T4 GPU) for faster tuning.
> The pipeline is CPU-compatible but XGBoost tuning is 5-10x faster on GPU.

## Setup

In [ ]:
%cd /content
!git clone https://github.com/moneet-dev/nanowire-ml.git 2>/dev/null || (cd nanowire-ml && git pull)
%cd /content/nanowire-ml

!pip install -q scikit-learn xgboost biopython shap matplotlib seaborn 2>/dev/null
!pip install -q --no-deps iFeatureOmegaCLI
!pip install -q networkx rdkit 2>/dev/null

# ESM-2 (use Colab's pre-installed torch; only install fair-esm)
!pip install -q fair-esm 2>/dev/null

# Verify GPU
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    print(f'\nGPU: {gpu} ({torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB)')
else:
    print('\nWARNING: No GPU. Switch to T4: Runtime > Change runtime type > T4 GPU')

print('=== Setup complete ===')

In [2]:
# Mount Google Drive and symlink features directory for persistence
from google.colab import drive
import os

drive.mount('/content/drive')
feat_dir = '/content/drive/MyDrive/nanowire-ml/features'
os.makedirs(feat_dir, exist_ok=True)

local_feat = 'data/features'
if os.path.islink(local_feat):
    os.unlink(local_feat)
elif os.path.isdir(local_feat):
    import shutil
    shutil.rmtree(local_feat)
os.symlink(feat_dir, local_feat)

print(f'Features dir -> {feat_dir}')

Mounted at /content/drive
Features dir -> /content/drive/MyDrive/nanowire-ml/features


## Phase 1 — Data Preparation

In [3]:
!python scripts/01_data_prep.py

Phase 1 - data preparation complete

  non-redundant : data/raw/merged.fasta, data/raw/labels.csv
  redundant     : data/raw/merged_redundant.fasta, data/raw/labels_redundant.csv
  sanity CSV    : results/metrics/data_sanity.csv

  n_total                           : 1664
  n_positive                        : 847
  n_negative                        : 817
  n_orphan_headers_repaired         : 1
  n_duplicate_sequences_dropped     : 333
  n_unique_sequences                : 1664
  len_positive_min                  : 40
  len_positive_median               : 135.0
  len_positive_max                  : 768
  len_positive_mean                 : 182.8
  aromatic_frac_positive_mean       : 0.0776
  len_negative_min                  : 72
  len_negative_median               : 344.0
  len_negative_max                  : 2054
  len_negative_mean                 : 353.4
  aromatic_frac_negative_mean       : 0.0551
  n_seqs_with_noncanonical_residues : 8
  redundant_n_total                 : 1997
  

## Phase 2 — Feature Extraction

Extracts 22 iFeature descriptor groups. First run takes ~3 min; subsequent runs skip if `master_features.csv` already exists on Drive.

In [4]:
import os

nr_exists = os.path.exists('data/features/master_features.csv')
red_exists = os.path.exists('data/features/master_features_redundant.csv')

if nr_exists and red_exists:
    print('Both feature matrices already exist on Drive — skipping extraction.')
else:
    variants = []
    if not nr_exists:
        variants.append('nr')
    if not red_exists:
        variants.append('redundant')
    cmd = 'python scripts/02_feature_extraction.py ' + ' '.join(variants)
    print(f'Running: {cmd}')
    !{cmd}

Running: python scripts/02_feature_extraction.py nr redundant

=== variant 'nr': merged.fasta ===
[ok]   AAC              ->    20 features
[ok]   GAAC             ->     5 features
The descriptor type does not exist.
[ok]   CKSAAP type 1    ->     5 features
The descriptor type does not exist.
[ok]   DPC type 1       ->     5 features
The descriptor type does not exist.
[ok]   DPC type 2       ->     5 features
The descriptor type does not exist.
[ok]   TPC type 1       ->     5 features
The descriptor type does not exist.
[ok]   TPC type 2       ->     5 features
[ok]   CTDC             ->    39 features
[ok]   CTDT             ->    39 features
[ok]   CTDD             ->   195 features
[ok]   CTriad           ->   343 features
The descriptor type does not exist.
[ok]   KNN              ->   343 features
[ok]   Geary            ->    24 features
[ok]   Moran            ->    24 features
[ok]   NMBroto          ->    24 features
[ok]   AC               ->    24 features
[ok]   CC     

## Phase 3 — Reproduce Baseline (Raya et al. 2025 Table 2)

In [5]:
# Reproduce on redundant (paper-faithful) set
!python scripts/03_reproduce_baseline.py redundant

variant 'redundant': X=(1997, 964), pos=998, neg=999

  SVM      acc=0.9933 (paper 0.9487)  roc_auc=0.9998 (paper 0.9696)  pr_auc=0.9998  ece=0.0074  [1s]
  RF       acc=0.9983 (paper 0.9669)  roc_auc=1.0000 (paper 0.9826)  pr_auc=1.0000  ece=0.0361  [2s]
  XGBoost  acc=0.9967 (paper 0.9665)  roc_auc=0.9999 (paper 0.9857)  pr_auc=0.9999  ece=0.0052  [1s]
  LR       acc=0.9967 (paper 0.9605)  roc_auc=1.0000 (paper 0.9903)  pr_auc=1.0000  ece=0.0038  [0s]
  MLP      acc=0.9950 (paper 0.9613)  roc_auc=1.0000 (paper 0.992)  pr_auc=1.0000  ece=0.0046  [1s]

wrote results/metrics/reproduction_table.csv


In [6]:
# Reproduce on non-redundant set
!python scripts/03_reproduce_baseline.py nr

variant 'nr': X=(1664, 964), pos=847, neg=817

  SVM      acc=0.9940 (paper 0.9487)  roc_auc=0.9998 (paper 0.9696)  pr_auc=0.9998  ece=0.0097  [1s]
  RF       acc=0.9920 (paper 0.9669)  roc_auc=0.9999 (paper 0.9826)  pr_auc=0.9999  ece=0.0355  [1s]
  XGBoost  acc=0.9940 (paper 0.9665)  roc_auc=0.9996 (paper 0.9857)  pr_auc=0.9997  ece=0.0075  [1s]
  LR       acc=0.9980 (paper 0.9605)  roc_auc=1.0000 (paper 0.9903)  pr_auc=1.0000  ece=0.0029  [0s]
  MLP      acc=0.9940 (paper 0.9613)  roc_auc=0.9997 (paper 0.992)  pr_auc=0.9997  ece=0.0053  [1s]

wrote results/metrics/reproduction_table_nr.csv


In [7]:
# Display reproduction comparison
import pandas as pd

print('=== Redundant (paper-faithful) ===')
display(pd.read_csv('results/metrics/reproduction_table.csv'))

print('\n=== Non-redundant ===')
display(pd.read_csv('results/metrics/reproduction_table_nr.csv'))

=== Redundant (paper-faithful) ===


,model,paper_acc,test_acc,test_precision,test_recall,test_f1,paper_roc_auc,test_roc_auc,test_pr_auc,test_ece,fit_seconds
0,SVM,0.9487,0.9933,0.9901,0.9967,0.9934,0.9696,0.9998,0.9998,0.0074,0.8
1,RF,0.9669,0.9983,1.0000,0.9967,0.9983,0.9826,1.0000,1.0000,0.0361,1.7
2,XGBoost,0.9665,0.9967,0.9967,0.9967,0.9967,0.9857,0.9999,0.9999,0.0052,1.3
3,LR,0.9605,0.9967,1.0000,0.9933,0.9967,0.9903,1.0000,1.0000,0.0038,0.1
4,MLP,0.9613,0.9950,1.0000,0.9900,0.9950,0.9920,1.0000,1.0000,0.0046,0.6



=== Non-redundant ===


,model,paper_acc,test_acc,test_precision,test_recall,test_f1,paper_roc_auc,test_roc_auc,test_pr_auc,test_ece,fit_seconds
0,SVM,0.9487,0.994,0.9884,1.0000,0.9942,0.9696,0.9998,0.9998,0.0097,0.7
1,RF,0.9669,0.992,0.9922,0.9922,0.9922,0.9826,0.9999,0.9999,0.0355,0.8
2,XGBoost,0.9665,0.994,0.9961,0.9922,0.9941,0.9857,0.9996,0.9997,0.0075,0.9
3,LR,0.9605,0.998,1.0000,0.9961,0.9980,0.9903,1.0000,1.0000,0.0029,0.1
4,MLP,0.9613,0.994,0.9961,0.9922,0.9941,0.9920,0.9997,0.9997,0.0053,0.5


## Phase 4 — Extended Pipeline

Novel contributions: hyperparameter tuning (RandomizedSearchCV, 50 iter), calibration (Platt + isotonic), stacking ensemble, SHAP attribution, bootstrap CIs.

**This is the longest cell** — ~15 min on Colab CPU, ~5 min on T4 GPU.

In [ ]:
!python -u scripts/04_extended_pipeline.py nr

variant 'nr': train=(1164, 964), test=(500, 964)

=== Hyperparameter tuning ===
  tuning RF (50 iter)...

In [ ]:
# Display extended results
import pandas as pd
print('=== Extended Pipeline Results ===')
display(pd.read_csv('results/metrics/extended_table.csv'))

print('\n=== Best Hyperparameters ===')
import json
with open('results/metrics/best_params.json') as f:
    print(json.dumps(json.load(f), indent=2))

In [ ]:
# Display figures
from IPython.display import Image, display
import os

fig_dir = 'results/figures'
for fig in ['roc_curves.png', 'pr_curves.png', 'calibration_plots.png', 'shap_summary.png']:
    path = os.path.join(fig_dir, fig)
    if os.path.exists(path):
        print(f'\n--- {fig} ---')
        display(Image(filename=path, width=700))

## Phase 5 — ESM-2 Protein Language Model

Generate 1280-dim ESM-2 embeddings and run the same classifier battery. Produces the **iFeature vs ESM-2 comparison** — a core research finding. Requires T4 GPU (~5 min for embedding + classifiers).

In [ ]:
!python -u scripts/04b_esm2_pipeline.py

In [ ]:
# Display ESM-2 results and iFeature vs ESM-2 comparison
import pandas as pd
from IPython.display import Image, display
import os

if os.path.exists('results/metrics/esm2_table.csv'):
    print('=== ESM-2 Classifier Results ===')
    display(pd.read_csv('results/metrics/esm2_table.csv'))

if os.path.exists('results/metrics/esm2_vs_ifeature.csv'):
    print('\n=== ESM-2 vs iFeature Comparison ===')
    display(pd.read_csv('results/metrics/esm2_vs_ifeature.csv'))

if os.path.exists('results/figures/esm2_roc_pr_curves.png'):
    display(Image(filename='results/figures/esm2_roc_pr_curves.png', width=700))

## Phase 6 — Feature Ablation Study

Tests each descriptor group individually and in biologically motivated combinations.

In [ ]:
!python -u scripts/05_ablation.py nr

In [ ]:
import pandas as pd
from IPython.display import Image, display

print('=== Ablation Results ===')
df = pd.read_csv('results/metrics/ablation_table.csv')
display(df.sort_values('roc_auc', ascending=False))

if os.path.exists('results/figures/ablation_bar.png'):
    display(Image(filename='results/figures/ablation_bar.png', width=700))

## Phase 7 — Autoresearch Loop

Runs 10 predefined experiments using the keep/revert pattern from Karpathy's autoresearch framework.

In [ ]:
!python -u scripts/06_autoresearch_loop.py

In [ ]:
import pandas as pd

print('=== Autoresearch Experiment Log ===')
display(pd.read_csv('results/metrics/autoresearch_log.csv'))

## Push Results to GitHub

Commit all result CSVs and push to the repo. You'll need a GitHub Personal Access Token (PAT) with `repo` scope.

**Create one at:** Settings → Developer settings → Personal access tokens → Generate new token (classic) → check `repo` scope.

In [ ]:
from getpass import getpass
token = getpass('GitHub Personal Access Token (repo scope): ')

!git config user.email "devadigamoneet@gmail.com"
!git config user.name "moneet-dev"

import subprocess
subprocess.run(['git', 'remote', 'set-url', 'origin',
                f'https://{token}@github.com/moneet-dev/nanowire-ml.git'],
               capture_output=True)

!git add results/metrics/ scripts/ src/ notebooks/
!git status

!git commit -m "results: Phases 3-7 from Colab — reproduction, tuning, calibration, SHAP, ablation, autoresearch"
!git push origin main

# Clear token from remote URL
!git remote set-url origin https://github.com/moneet-dev/nanowire-ml.git
print('\n=== Pushed to GitHub ===')

## Summary

All phases complete. Check the results:

In [ ]:
import os
print('=== Result files ===')
for f in sorted(os.listdir('results/metrics')):
    if f.endswith('.csv') or f.endswith('.json'):
        size = os.path.getsize(f'results/metrics/{f}')
        print(f'  {f:45s} {size:>8,d} bytes')

print('\n=== Figures ===')
fig_dir = 'results/figures'
if os.path.exists(fig_dir):
    for f in sorted(os.listdir(fig_dir)):
        if f.endswith('.png'):
            size = os.path.getsize(f'{fig_dir}/{f}')
            print(f'  {f:45s} {size:>8,d} bytes')